# Lesson 3 — Speculative fan-out: ask everything, let code decide

Lesson 2 covered `Score` and `Noul`, and how to pick the right primitive. This lesson is about composing several questions — of any primitive type — into one request, including ones you might not end up needing.

Ask your agent if anything's unclear.

<details>
  <summary><strong>Primary source:</strong> docs.typesafe.ai/patterns/fan-out (click to expand)</summary>

# Speculative fan-out

> Send many questions in a single call, including speculative ones, and let your code decide what's relevant.

Because TypeSafe supports sending many questions in a single API call, we recommend putting all of the questions your system needs in a single request, and then using code to decide what is relevant after the fact. All questions are evaluated in parallel, so adding more questions to a call typically doesn't add any latency to the response.

## Example: support ticket triage

Let's imagine you are building a support system that needs to triage support tickets. You need to classify the ticket into a category. If it's a bug report, you also need to determine the severity of the bug.

Instead of asking for the category first and then the severity in a follow-up call, you can ask for both at the same time. If the ticket is not a bug report, you simply ignore the results of the bug severity question.

### Step 1: speculative fan-out

```json
{
  "state": "Hi, I placed an order (#98423) last Thursday and was charged twice. I also can't log in after the site update, and adding Apple Pay would be really helpful. This is getting frustrating.",
  "questions": {
    "category": {
      "type": "choice",
      "instructions": "Determine the broad category of this support ticket",
      "criteria": {
        "bug_report": "The user is reporting something that is broken or producing errors",
        "billing": "Charges, invoices, refunds, subscriptions",
        "feature_request": "The user is requesting new functionality",
        "account": "Login, permissions, profile, security"
      }
    },
    "bug_severity": {
      "type": "score",
      "instructions": "How severe is the reported issue",
      "criteria": [
        "Cosmetic; no impact to functionality",
        "Broken or degraded feature; workaround exists",
        "Blocking issue; no workaround exists"
      ]
    },
    "has_reproducible_steps": {
      "type": "noul",
      "instructions": "The user describes specific steps to reproduce the issue"
    },
    "refund_requested": {
      "type": "noul",
      "instructions": "The user is explicitly asking for a refund or credit"
    },
    "frustration": {
      "type": "score",
      "instructions": "How frustrated the user appears",
      "criteria": ["Calm, matter-of-fact", "Frustrated but civil", "Very angry"]
    }
  }
}
```

> **Speculative questions:** `bug_severity` and `has_reproducible_steps` only matter if the ticket is a bug report. `refund_requested` only matters for billing. We include all upfront because there is no speed cost for additional questions. If the ticket turns out to be a feature request, the bug severity result will be irrelevant, in which case your code path simply ignores it.

### Step 2: route with code

Your code decides what is relevant based on the classification result:

```python
# triage.py
category = response.answers["category"]
bug_severity = response.answers["bug_severity"]
bug_repro = response.answers["has_reproducible_steps"]
refund = response.answers["refund_requested"]
frustration = response.answers["frustration"]

if category.choice == "bug_report":
    if bug_severity.score > 1.5 and bug_repro.noul > 0.6:
        escalate_to_engineering(ticket_id, severity="high")
    else:
        add_to_bug_backlog(ticket_id)

elif category.choice == "billing":
    if refund.noul > 0.7:
        route_to_billing_with_flag(ticket_id, refund_likely=True)
    else:
        route_to_billing(ticket_id)

elif category.choice == "feature_request":
    log_feature_request(ticket_id)

# Frustration is useful regardless of category
if frustration.score > 1.5:
    flag_for_priority_response(ticket_id)
```

Everything needed for the full decision tree comes from one call. Speculative questions are ignored when irrelevant and save a round trip when they are not.

</details>

## The core idea

Every question in a request is evaluated independently and in parallel — one answer is never hidden context for another. We've leaned on this since lesson 2, asking a `Score` and two `Noul` questions together in one call. Speculative fan-out just names the strategy of pushing that all the way: instead of round-tripping ("what category is this?" → wait → "okay, how severe is the bug?"), ask every question your decision tree could possibly need in the *first* call, then let ordinary code decide which answers actually matter.

The reason this works: adding a question costs a few extra tokens, not extra latency. A question you end up not needing is close to free to have asked.

In [ ]:
import { client } from "../assets/typesafe-client.ts";
import { choice, score, noul } from "npm:@typesafe-ai/sdk";

const response = await client.systemOne({
  state:
    "Hi, I placed an order (#98423) last Thursday and was charged twice. I also can't log in after the site update, and adding Apple Pay would be really helpful. This is getting frustrating.",
  questions: {
    category: choice("Determine the broad category of this support ticket", {
      bug_report: "The user is reporting something that is broken or producing errors",
      billing: "Charges, invoices, refunds, subscriptions",
      feature_request: "The user is requesting new functionality",
      account: "Login, permissions, profile, security",
    }),
    bugSeverity: score("How severe is the reported issue", [
      "Cosmetic; no impact to functionality",
      "Broken or degraded feature; workaround exists",
      "Blocking issue; no workaround exists",
    ]),
    hasReproducibleSteps: noul("The user describes specific steps to reproduce the issue"),
    refundRequested: noul("The user is explicitly asking for a refund or credit"),
    frustration: score("How frustrated the user appears", [
      "Calm, matter-of-fact",
      "Frustrated but civil",
      "Very angry",
    ]),
  },
});

const { category, bugSeverity, hasReproducibleSteps, refundRequested, frustration } =
  response.answers;

console.log("category:", category.choice);

if (category.choice === "bug_report") {
  console.log(
    bugSeverity.score > 1.5 && hasReproducibleSteps.noul > 0.6
      ? "→ escalate to engineering (high severity, reproducible)"
      : "→ add to bug backlog",
  );
} else if (category.choice === "billing") {
  console.log(
    refundRequested.noul > 0.7
      ? "→ route to billing, flag as refund-likely"
      : "→ route to billing",
  );
} else if (category.choice === "feature_request") {
  console.log("→ log feature request");
} else {
  console.log("→ route to account team");
}

if (frustration.score > 1.5) {
  console.log("→ also flag for priority response");
}

## What just happened

Five typed answers from one call — a `Choice`, two `Score`s, and two `Noul`s, all evaluated against the same ticket. This ticket is deliberately messy: a duplicate charge (billing), a login problem (account), *and* a feature request, all in one message. Whichever category wins, the other questions' answers are still sitting there, unused but harmless — that's the whole point of asking them speculatively.

**Try it yourself:** predict which `category` wins before you run it, given the ticket mixes three plausible categories. Then try rewriting the ticket as an unambiguous bug report with clear repro steps, and predict whether it escalates to engineering or lands in the backlog.

## Next up

Next lesson: Composite scoring — splitting one complex judgment into several weighted `Score` questions and combining them in code, the pattern we deliberately trimmed out of lesson 2's embedded Score source.

Questions? Ask your agent — it has the same research this lesson was built from, in `research/typesafe-ai.md`.